# 03 — Descriptive Analysis
Quintile bins, Table A & B, delta_shot_real, delta_goal_real, Fig 2 & 3.

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import os

from config import INTERIM_DIR, TABLES_DIR
from utils import make_quintile_bins, compute_delta, summarize_momentum_table
from plots import plot_momentum_bins

os.makedirs(TABLES_DIR, exist_ok=True)

In [2]:
# Load interim data
team_windows_df = pd.read_csv(os.path.join(INTERIM_DIR, 'team_windows.csv'))
print(f'Loaded team_windows_df: {len(team_windows_df)} rows')

Loaded team_windows_df: 10368 rows


In [3]:
# Quintile bins
team_windows_df['quintile_bin'] = make_quintile_bins(team_windows_df['rolling_xg_5'])
print('Quintile bin distribution:')
print(team_windows_df['quintile_bin'].value_counts().sort_index())

Quintile bin distribution:
quintile_bin
0    8295
1    2073
Name: count, dtype: int64


In [4]:
# Table A: momentum summary
table_a = summarize_momentum_table(team_windows_df)
print('=== Table A: Momentum Summary ===')
print(table_a.to_string(index=False))

table_a.to_csv(os.path.join(TABLES_DIR, 'table_a_momentum.csv'), index=False)
print('\nSaved to outputs/tables/table_a_momentum.csv')

=== Table A: Momentum Summary ===
 quintile_bin  n_rows  rolling_xg_5_mean  P_shot_next_2  P_goal_next_5
            0    8295           0.008761       0.180229       0.057143
            1    2073           0.264963       0.209358       0.097443

Saved to outputs/tables/table_a_momentum.csv


In [5]:
# Table B: breakdown by game state
table_b_rows = []
for gs, grp in team_windows_df.groupby('game_state'):
    grp = grp.copy()
    grp['qbin'] = make_quintile_bins(grp['rolling_xg_5'])
    agg = grp.groupby('qbin').agg(
        n=('shot_next_2','count'),
        P_shot_next_2=('shot_next_2','mean'),
        P_goal_next_5=('goal_next_5','mean'),
    ).reset_index()
    agg['game_state'] = gs
    table_b_rows.append(agg)

table_b = pd.concat(table_b_rows, ignore_index=True)
print('=== Table B: Momentum by Game State ===')
print(table_b.to_string(index=False))
table_b.to_csv(os.path.join(TABLES_DIR, 'table_b_by_game_state.csv'), index=False)

=== Table B: Momentum by Game State ===
 qbin    n  P_shot_next_2  P_goal_next_5 game_state
    0 1463       0.176350       0.087491    leading
    1  486       0.193416       0.090535    leading
    2  484       0.185950       0.103306    leading
    0 4403       0.174427       0.046559       tied
    1 1099       0.232939       0.093722       tied
    0 1947       0.179250       0.052388   trailing
    1  486       0.234568       0.090535   trailing


In [6]:
# Compute deltas
delta_shot_real = compute_delta(team_windows_df, 'shot_next_2')
delta_goal_real = compute_delta(team_windows_df, 'goal_next_5')
print(f'delta_shot_real (P(Q5) - P(Q1)): {delta_shot_real:.4f}')
print(f'delta_goal_real (P(Q5) - P(Q1)): {delta_goal_real:.4f}')

delta_shot_real (P(Q5) - P(Q1)): 0.0291
delta_goal_real (P(Q5) - P(Q1)): 0.0403


In [7]:
# Fig 2 & 3: Momentum bin plots
plot_momentum_bins(team_windows_df)

Saved → /Users/aidenpark/Project/cs109-soccer-momentum/outputs/figures/fig2_momentum_bins.png
